In [19]:
import numpy as np

eps = 0.0000000001
inf = 1000000000000000000000000000000

class SimplexMethod:
    def sparse_matmul(self, Q: np.ndarray, A_inv: np.ndarray, i: int) -> np.ndarray:
        Ans = np.zeros(shape=Q.shape)
        for k in range(Q.shape[0]):
            for j in range(Q.shape[0]):
                if k == i:
                    Ans[k][j] = Q[k][i] * A_inv[i][j]
                else:
                    Ans[k][j] = Q[k][i] * A_inv[i][j] + A_inv[k][j]
        
        return Ans

    def inverse_x(self, A_inv: np.ndarray, x: np.ndarray, i: int) -> np.ndarray:
        l = A_inv @ x
        if np.abs(l[i]) < eps:
            return None
        l_i = l[i]
        l[i] = -1
        l /= -l_i
        Q = np.eye(x.shape[0])
        Q[:, i] = l
        return self.sparse_matmul(Q, A_inv, i)
    
    def to_canonical_form(
        self,
        c: list[float],
        d: int,
        A: list[list[float]],
        b: list[float],
        r: list[int],
        sigma: list[int]
    ):
        # step 1
        if d == -1:
            c = [el * -1.0 for el in c]
            d = 1
        
        # step 2
        for i, sign in enumerate(r):
            if sign == -1:
                for ind, row in enumerate(A):
                    if ind == i:
                        row.insert(len(row), 1.0)
                        continue
                    row.insert(len(row), 0.0)
                c.insert(len(c), 0.0)
                r[i] = 0
                sigma.insert(len(sigma), 1)
        
        # step 3
        for i, sign in enumerate(r):
            if sign == 1:
                for ind, row in enumerate(A):
                    if ind == i:
                        row.insert(len(row), -1.0)
                        continue
                    row.insert(len(row), 0.0)
                c.insert(len(c), 0.0)
                r[i] = 0
                sigma.insert(len(sigma), 1)
        
        # step 4
        for i, sign in enumerate(sigma):
            if sign == -1:
                for row in A:
                    row[i] = -row[i]
                c[i] = -c[i]
                sigma[i] = 1
        
        # step 5
        for i, sign in enumerate(sigma):
            if sign == 0:
                for row in A:
                    row.insert(i + 1, -row[i])
                c.insert(i + 1, -c[i])
                sigma[i] = 1
                sigma.insert(i + 1, 1)
        
        return c, A, b

    def simplex_main_phase(
        self,
        p_c: np.ndarray,
        p_A: np.ndarray,
        p_x: np.ndarray,
        p_B: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
        c = np.array(p_c, dtype=float)
        A = np.array(p_A, dtype=float)
        x = np.array(p_x, dtype=float)
        B = np.array(p_B, dtype=int)
        is_first_iteration = True
        A_B_inv = None

        while True:
            # step 1
            A_B = A[:, B]

            if is_first_iteration:
                A_B_inv = np.linalg.inv(A_B)
                is_first_iteration = False
            else:
                A_B_inv = self.inverse_x(A_B_inv, A[:, B[k]], k)
            
            # step 2
            c_B = c[B]

            # step 3
            u = A_B_inv.T @ c_B

            # step 4
            delta = c - A.T @ u

            # step 5
            if np.max(delta) <= eps:
                return x, B
            
            # step 6
            j0 = np.argmax(delta > eps)

            # step 7
            z = A_B_inv @ A[:, j0]

            # step 8
            theta = np.array([x[B[i]] / z[i] if z[i] > 0 else inf for i in range(z.shape[0])])

            # step 9
            theta0 = np.min(theta)

            # step 10
            if theta0 == inf:
                print("target function is not bounded (no optimal plan)")
                return None, None

            # step 11
            k = np.argmin(theta)

            # step 12
            j_so_zvezdochkoy = B[k]
            B[k] = j0

            # step 13
            x[j0] = theta0
            x[j_so_zvezdochkoy] = 0
            for i in range(B.shape[0]):
                if B[i] == j0:
                    continue
                x[B[i]] = x[B[i]] - theta0 * z[i]

    def simplex_init_phase(
        self,
        p_c: np.ndarray,
        p_A: np.ndarray,
        p_b: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray] | tuple[None, None, None, None]:
        c = np.array(p_c, dtype=float)
        A = np.array(p_A, dtype=float)
        b = np.array(p_b, dtype=float)
        m = A.shape[0]
        n = A.shape[1]

        # step 1
        for i in range(m):
            if b[i] < -eps:
                b[i] *= -1
                for j in range(n):
                    A[i][j] *= -1
        
        # step 2
        c_s_volnoy = np.zeros(shape=(n + m,), dtype=float)
        for i in range(n, n + m):
            c_s_volnoy[i] = -1
        A_s_volnoy = np.hstack([A, np.eye(m)])

        # step 3
        x_s_volnoy = np.zeros(shape=(n + m,), dtype=float)
        for i in range(m):
            x_s_volnoy[n + i] = b[i]
        B = np.array([el for el in range(n, n + m)], dtype=int)

        # step 4
        x_s_volnoy, B = self.simplex_main_phase(c_s_volnoy, A_s_volnoy, x_s_volnoy, B)
        
        # step 5
        sovmestna = True
        for i in range(n, n + m):
            if np.abs(x_s_volnoy[i]) > eps:
                sovmestna = False
                break
        
        if not sovmestna:
            print("Task has no solutions")
            return None, None, None, None
        
        # step 6
        x = np.zeros(shape=(n,), dtype=float)
        for i in range(n):
            x[i] = x_s_volnoy[i]
        
        while (True):
            # step 7
            B_podhodit = True
            for i in range(m):
                if not (0 <= B[i] < n):
                    B_podhodit = False
                    break
            
            if B_podhodit:
                return x, B, A, b
            
            # step 8
            k = max((idx for idx in range(m) if B[idx] >= n), key=lambda idx: B[idx])
            jk = int(B[k])
            i = jk - n
            
            # step 9
            A_B_s_volnoy = np.zeros(shape=(m, m), dtype=float)
            for ind in range(m):
                for row_ind in range(m):
                    A_B_s_volnoy[row_ind][ind] = A_s_volnoy[row_ind][B[ind]]
            A_B_s_volnoy_inv = np.linalg.inv(A_B_s_volnoy)
            
            changed = False
            for j in range(n):
                if j in B:
                    continue
                
                lj = A_B_s_volnoy_inv @ A_s_volnoy[:, j]
                
                if np.abs(lj[k]) > eps:
                    B[k] = j
                    changed = True
                    break
            
            if not changed:
                A = np.delete(A, i, axis=0)
                b = np.delete(b, i)
                B = np.delete(B, k)
                A_s_volnoy = np.delete(A_s_volnoy, i, axis=0)
                m -= 1

    def solve(
        self,
        p_c: list[float],
        p_A: list[list[float]],
        p_b: list[float],
        p_d: int = None,
        p_r: list[int] = None,
        p_sigma: list[int] = None
    ) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
        '''
        p_c : vector of costs in optimization function
        p_A : matrix of main constraints Ax = b
        p_b : vector of main constraints Ax = b
        p_d : optimization direction: 1 - max; -1 - min if task is not in canonical form
        p_r : signs of main constraints if task is not in canonical form
        p_sigma : signs of x if task is not in canonical form
        '''

        if (p_d is None and p_r is None and p_sigma is None):
            # solve problem in canonical form
            x, B, A, b = self.simplex_init_phase(p_c, p_A, p_b)
            if (x is None and B is None and A is None and b is None):
                return None, None
            x_sol, B_sol = self.simplex_main_phase(p_c, A, x, B)
            return x_sol, B_sol
        else:
            # problem is not in canonical form
            c_can, A_can, b_can = self.to_canonical_form(p_c, p_d, p_A, p_b, p_r, p_sigma)
            x, B, A_can, b_can = self.simplex_init_phase(c_can, A_can, b_can)
            if (x is None and B is None and A is None and b is None):
                return None, None
            x_sol, B_sol = self.simplex_main_phase(c_can, A_can, x, B)
            return x_sol, B_sol


In [20]:
# TESTS

solver = SimplexMethod()



print("\nTEST 1")
c = [1, 0, 0]
A = [[1, 1, 1],
     [2, 2, 2]]
b = [0, 0]

x, B, A, b = solver.simplex_init_phase(c, A, b)
print(f"x = {x}")
print(f"B = {B}")
print(f"A = {A}")
print(f"b = {b}")



print("\nTEST 2")
c = [20, 25]
d = 1
A = [[1, 2],
     [2, 1],
     [100, 0],
     [0, 100]]
b = [10, 11, 500, 400]
r = [-1, -1, -1, -1]
sigma = [1, 1]

x_sol, B_sol = solver.solve(c, A, b, d, r, sigma)

print(f"x_sol = {x_sol}")
print(f"B_sol = {B_sol}")



print("\nTEST 3")
c = [-8, -4, -1, -8, -4, -3, -9, -7, -5]
A = [[1, 1, 1, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 1, 1, 1, 0, 0, 0],
     [0, 0, 0, 0, 0, 0, 1, 1, 1],
     [1, 0, 0, 1, 0, 0, 1, 0, 0],
     [0, 1, 0, 0, 1, 0, 0, 1, 0],
     [0, 0, 1, 0, 0, 1, 0, 0, 1]]
b = [100, 300, 300, 300, 200, 200]
x_sol, B_sol = solver.solve(c, A, b)

print(f"x_sol = {x_sol}")
print(f"B_sol = {B_sol}")



print("\nTEST 4")
c = [1]
A = [[1],
     [-1]]
b = [10, -9]
x_sol, B_sol = solver.solve(c, A, b)

print(f"x_sol = {x_sol}")
print(f"B_sol = {B_sol}")


TEST 1
x = [0. 0. 0.]
B = [0]
A = [[1. 1. 1.]]
b = [0.]

TEST 2
x_sol = [  4.   3.   0.   0. 100. 100.]
B_sol = [4 1 0 5]

TEST 3
x_sol = [  0.   0. 100.   0. 200. 100. 300.   0.   0.]
B_sol = [2 4 5 3 6]

TEST 4
Task has no solutions
x_sol = None
B_sol = None


In [ ]:
solver = SimplexMethod()

def Gomory_constraint(
    p_c: list[float],
    p_A: list[list[float]],
    p_b: list[float]
):
    # step 1
    x_sol, B_sol = solver.solve(p_c, p_A, p_b)

    # step 2
    if (x_sol is None and B_sol is None):
        return None, None

    # step 3
    is_all_int = np.all(np.mod(x_sol, 1) == 0)
    if is_all_int:
        return x_sol

    # step 4
    # need to create constraint Gomory

    # step 5
    B_list = list(B_sol)
    frac_indices = [i for i in B_list if not np.isclose(x_sol[i], round(x_sol[i]))]
    i = frac_indices[0]
    k = B_list.index(i)

    # step 6
    n = len(p_c)
    A = np.array(p_A, dtype=float)
    B_set = set(B_list)
    N_indices = [j for j in range(n) if j not in B_set]
    A_B = A[:, B_list]
    A_N = A[:, N_indices]

    # step 7
    # it is not nessesary

    # step 8
    A_B_inv = np.linalg.inv(A_B)

    # step 9
    Q = A_B_inv @ A_N

    # step 10
    l = Q[k]

    # step 11
    frac_l = l - np.floor(l)
    frac_xi = x_sol[i] - np.floor(x_sol[i])

    gomory = np.zeros(n + 1)
    for p, j in enumerate(N_indices):
        gomory[j] = frac_l[p]
    gomory[-1] = -1.0

    rhs = frac_xi

    return gomory, rhs

    

In [25]:
c = [0, 1, 0, 0]
A = [[3, 2, 1, 0],
     [-3, 2, 0, 1]]
b = [6, 0]

gomory, rhs = Gomory_constraint(c, A, b)

print(f"gomory = {gomory}, rhs = {rhs}")

gomory = [ 0.    0.    0.25  0.25 -1.  ], rhs = 0.5
